<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Assignment 4 — The Diet Problem at Your Restaurant

In Studio 3 you fed a football team at McDonald's. Now you do it for **your own restaurant**, the one
you were assigned in class.

**You do not write any model code.** The model from class is already here. You work through six short
steps, top to bottom. In each step you run a cell, look at what it prints, and fill in a few blanks.

| Step | What you do | Points |
|---|---|---|
| 1 | Tell the notebook your name and your restaurant | - |
| 2 | Load the model (just run it) | - |
| 3 | Find the cheapest order | 20 |
| 4 | Try two scenarios | 20 |
| 5 | Find the shadow price of salt | 45 |
| 6 | Make a recommendation | 15 |

**The only cells you type in** are the ones marked `<-- CHANGE` and the ones that say **YOUR ANSWER**.
Leave everything else alone.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Step 1 · Your name and your restaurant

Type your name. Then find your restaurant in this list and copy its file name **exactly**.

| Restaurant | File name | Restaurant | File name |
|---|---|---|---|
| Arby's | `arbys_menu.csv` | Panera Bread | `panera_bread_menu.csv` |
| Bojangles | `bojangles_menu.csv` | Sonic | `sonic_menu.csv` |
| Burger King | `burger_king_menu.csv` | Starbucks | `starbucks_menu.csv` |
| Chick-fil-A | `chick_fil_a_menu.csv` | Steak 'n Shake | `steak_n_shake_menu.csv` |
| Culver's | `culvers_menu.csv` | Subway | `subway_menu.csv` |
| Dairy Queen | `dairy_queen_menu.csv` | Taco Bell | `taco_bell_menu.csv` |
| Del Taco | `del_taco_menu.csv` | Wendy's | `wendys_menu.csv` |
| Hardee's | `hardees_menu.csv` | Whataburger | `whataburger_menu.csv` |
| Jack in the Box | `jack_in_the_box_menu.csv` | White Castle | `white_castle_menu.csv` |
| Krystal | `krystal_menu.csv` | Zaxby's | `zaxbys_menu.csv` |

In [ ]:
your_name = 'YOUR NAME HERE'            # <-- CHANGE: your name
menu_file = 'mcdonalds_menu.csv'        # <-- CHANGE: your restaurant's file name

# ---- nothing to change below this line ----
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'
menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + 'assignment_rules.csv')

print("Name:", your_name)
print("Menu file:", menu_file, "-", len(menu_df), "items")
if menu_file == 'mcdonalds_menu.csv':
    print()
    print("STOP! This is still the McDonald's menu from class. Change menu_file to YOUR restaurant.")
menu_df

That table is your menu. It is real: the nutrition comes from the New York City Health Department's
restaurant database (MenuStat, 2018) and the prices were listed in July and August 2026.

These are the rules for one player for one day. They are a little looser than the ones we used in
class, so that every restaurant has at least one order that works.

In [ ]:
rules_df

# Step 2 · Load the model

**Just run this cell.** You do not need to read it or change it.

It is the same team order model you ran in Studio 3.3 (53 players, one day of meals). The only
difference is that it is wrapped up so that you can run a whole scenario with **one line**:
`run_order("a name for the scenario")`.

In [ ]:
# ---------- THE MODEL (run this cell - do not change it) ----------
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)    # keeps the output tidy when an order is infeasible

# the menu items - every column after 'price' is a nutrient
items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

# price[item] and nutrition[item][nutrient] (for ONE of that item)
price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

# a readable label for each nutrient
label = {}
for nutrient in nutrient_columns:
    label[nutrient] = nutrient
label['carbs'] = 'Carbohydrates (g)'

# the rules per player: one dictionary of minimums, one of maximums
player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    label[nutrient] = rules_df.loc[i, 'label']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

team_size = 53        # players on the roster
scoreboard = {}       # scenario name -> cost per player


def run_order(scenario, max_of_one_item=4, must_have=None, sodium_limit=None, report=True):
    # the rules for this run (a fresh copy every time)
    low = dict(player_min)
    high = dict(player_max)
    if sodium_limit is not None:
        high['sodium'] = int(sodium_limit)
    if must_have is None:
        must_have = {}

    # ---- the model: exactly what we built in class ----
    model = ConcreteModel()
    model.x = Var(items, domain=NonNegativeReals)

    obj_expr = 0
    for item in items:
        obj_expr += price[item] * model.x[item]
    model.cost = Objective(expr = obj_expr, sense = minimize)

    model.constraints = ConstraintList()
    for nutrient in low:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
    for nutrient in high:
        nutrient_expr = 0
        for item in items:
            nutrient_expr += nutrition[item][nutrient] * model.x[item]
        model.constraints.add(nutrient_expr <= high[nutrient] * team_size)
    for item in items:
        model.constraints.add(model.x[item] <= max_of_one_item * team_size)
    for item in must_have:
        model.constraints.add(model.x[item] >= must_have[item] * team_size)

    results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

    # ---- no order works ----
    if results.solver.termination_condition != TerminationCondition.optimal:
        if report:
            print("SCENARIO:", scenario)
            print("INFEASIBLE! No order from this menu can satisfy every rule at the same time.")
        return np.nan

    per_player = model.cost() / team_size
    if not report:
        return per_player

    # ---- the report ----
    print("SCENARIO:", scenario)
    print("Cost per player = $%.2f" % per_player)
    print("Cost for the team of", team_size, "= $%.2f" % model.cost())

    print()
    print("The order, per player:")
    names = []
    counts = []
    for item in items:
        if model.x[item]() >= 0.5:          # the team orders at least half of one
            print("  ", round(model.x[item]() / team_size, 2), "x", item)
            names.append(item)
            counts.append(model.x[item]() / team_size)

    print()
    print("Nutrition per player, against the rules:")
    for nutrient in nutrient_columns:
        total = 0
        for item in items:
            total += nutrition[item][nutrient] * model.x[item]()
        amount = total / team_size
        note = ""
        if nutrient in low:
            note += " min " + str(low[nutrient])
            if abs(amount - low[nutrient]) < 0.5:
                note += "  <-- BINDING (sitting on the minimum)"
        if nutrient in high:
            note += " max " + str(high[nutrient])
            if abs(amount - high[nutrient]) < 0.5:
                note += "  <-- BINDING (sitting on the maximum)"
        if note == "":
            note = " no rule"
        print("  ", label[nutrient], ":", round(amount, 1), "|" + note)

    scoreboard[scenario] = round(per_player, 2)

    barh(names, counts, color='darkorange')
    xlabel('how many per player')
    title(scenario + ':  $%.2f per player' % per_player)
    tight_layout()
    show()
    return per_player


def shadow_price_at(limit):
    # reads two rows of the salt table (Step 5) and does the subtraction in words
    table = saltResults.set_index('Sodium limit (mg)')['Cost per player ($)']
    here = table.get(limit, np.nan)
    above = table.get(limit - 100, np.nan)
    print("Sodium limit:", limit, "mg")
    if pd.isna(here):
        print("   No order is possible at", limit, "mg. This menu cannot meet the rules with that little sodium.")
        return
    if pd.isna(above):
        print("   cost per player at", limit, "mg : $%.2f" % here)
        print("  ", limit, "mg is the lowest limit that works, so there is no row above it to compare with.")
        return
    above = round(above, 2)
    here = round(here, 2)
    saved = round(above - here, 2)
    print("   cost per player at", limit - 100, "mg : $%.2f" % above)
    print("   cost per player at", limit, "mg : $%.2f" % here)
    print("   shadow price of salt = %.2f - %.2f = $%.2f per player, per 100 mg" % (above, here, saved))
    if saved < 0.005:
        print("   -> $0.00 means the sodium rule is NOT binding here. Allowing more salt would not make the order any cheaper.")
    else:
        print("   -> Allowing 100 mg more sodium would save $%.2f per player, or $%.2f for the team of %d." % (saved, saved * team_size, team_size))
        print("   -> Cutting 100 mg of sodium would cost that much.")


print("Ready.", len(items), "menu items,", len(player_min) + len(player_max), "rules,", team_size, "players.")

# Step 3 · The cheapest order (20 points)

Run the cell. It finds the cheapest order that meets every rule, with **no more than 4 of any one item
per player**.

In [ ]:
base_cost = run_order("1 base")

### YOUR ANSWER - Step 3

Double-click this cell and fill in the blanks. Everything you need is printed right above.

**3a (5 points).** The cheapest order costs \$ \_\_\_\_\_\_ per player and \$ \_\_\_\_\_\_ for the team.

**3b (5 points).** The item the order uses the most of is \_\_\_\_\_\_\_\_\_\_\_\_ (\_\_\_\_ per player).

**3c (5 points).** A rule is **binding** when the order sits exactly on its minimum or maximum (the
report marks these for you). The binding rules at my restaurant are: \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_

**3d (5 points).** In one sentence: would you want to eat this for a day? Why or why not?

*your sentence*

# Step 4 · Two scenarios (20 points)

A scenario is the same model with one thing changed. Each one is a single line.

## Scenario 2 · No variety rule

What if a player could have as many of the same item as the solver likes? Just run it.

In [ ]:
no_variety_cost = run_order("2 no variety rule", max_of_one_item=100)

## Scenario 3 · The coach's order

The coach wants **every player to get one each of two items**. You are the coach: pick two items from
**your** menu. Copy the names exactly from the menu table in Step 1 (spelling and capitals matter).

**If it prints INFEASIBLE:** your two items break a rule when everybody has to eat both (usually too much
salt or fat). That is a real result, not a mistake. Pick two lighter items and run the cell again.

In [ ]:
item_1 = items[0]        # <-- CHANGE: replace items[0] with an item name in quotes, like 'Big Mac'
item_2 = items[1]        # <-- CHANGE: replace items[1] with a second item name in quotes

print("The coach's two items:", item_1, "and", item_2)
print()
coach_cost = run_order("3 coach's order", must_have={item_1 : 1, item_2 : 1})

## The scoreboard

Every scenario you ran is on the scoreboard. Run this cell to see them side by side.

In [ ]:
print(scoreboard)

scenario_names = []
dollars = []
for name in scoreboard:
    scenario_names.append(name)
    dollars.append(scoreboard[name])

barh(scenario_names, dollars, color='steelblue')
xlabel('$ per player')
title('What did each scenario cost?')
gca().invert_yaxis()
tight_layout()
show()

### YOUR ANSWER - Step 4

**4a (5 points).** Without the variety rule the order costs \$ \_\_\_\_\_\_ per player. So the variety rule
costs \$ \_\_\_\_\_\_ per player (base cost minus this one). If the cost did not change, write \$0.

**4b (5 points).** Without the variety rule, what does the order turn into? One sentence.

*your sentence*

**4c (5 points).** The coach's two items were \_\_\_\_\_\_\_\_\_\_\_\_ and \_\_\_\_\_\_\_\_\_\_\_\_. The order now
costs \$ \_\_\_\_\_\_ per player.

**4d (5 points).** The coach's order costs \$ \_\_\_\_\_\_ more per player than the base order. In one
sentence: is that worth it?

*your sentence*

# Step 5 · The shadow price of salt (45 points)

This is the most important step. Read this box twice.

> The rules say each player may have **at most 9,000 mg of sodium** a day. Cheap fast food is salty,
> so a sodium limit can cost money.
>
> **The shadow price of salt** answers one question:
> **if the sodium limit were 100 mg higher, how many dollars per player would the order get cheaper?**
>
> Turn it around and it is just as useful: **what does it cost, per player, to cut 100 mg of sodium?**

How do we find it? The same way we found the value of a carpentry hour in class: **solve the model
again and again, changing only the sodium limit**, and look at how the cost moves.

## 5.1 · Try every sodium limit

Run the cell. It solves your team order 81 times, with the sodium limit going from 2,000 mg up to
10,000 mg in steps of 100 mg. It takes a little while. Look at the loop: the **only** thing that
changes is `sodium_limit`.

In [ ]:
myRange = np.arange(2000, 10100, 100)    # the sodium limits to try
saltResults = pd.DataFrame()

for a in myRange:
    cost = run_order("salt", sodium_limit=a, report=False)    # the ONE thing we change is the sodium limit
    myX = pd.DataFrame([a, cost])
    myX = myX.T
    saltResults = pd.concat([saltResults, myX])

saltResults = saltResults.rename({0:"Sodium limit (mg)", 1:"Cost per player ($)"}, axis='columns')
saltResults.reset_index(drop=True, inplace=True)

# the shadow price: the cost in the row above (100 mg less sodium) minus the cost in this row
saltResults['Shadow price of salt ($ per 100 mg)'] = -saltResults['Cost per player ($)'].diff()
saltResults['Shadow price of salt ($ per 100 mg)'] = saltResults['Shadow price of salt ($ per 100 mg)'].round(3) + 0.0    # + 0.0 turns -0.0 into 0.0

# mark the base case (the 9,000 mg limit in this assignment)
saltResults['note'] = ''
saltResults.loc[saltResults['Sodium limit (mg)'] == 9000, 'note'] = '<-- base (9,000 mg)'

# the lowest limit that has an order at all
feasible = saltResults.dropna(subset=['Cost per player ($)'])
lowest_limit = int(feasible['Sodium limit (mg)'].min())
print("The lowest sodium limit this menu can meet is", lowest_limit, "mg.")
print("Below that the table shows NaN, which means no order is possible (infeasible).")

## 5.2 · Look at the table and the two pictures

Each row is 100 mg more sodium than the row above it. The last column is the shadow price:

**shadow price of salt = (cost in the row above) − (cost in this row)**

In [ ]:
# the table, starting a few rows before the first limit that works
pd.set_option('display.max_rows', 100)
saltResults[saltResults['Sodium limit (mg)'] >= lowest_limit - 300].round(2)

In [ ]:
base_limit = 9000      # the sodium limit in this assignment (the base case)

saltResults.plot(x='Sodium limit (mg)', y='Cost per player ($)')
axvline(base_limit, color='red', linewidth=3, label='base: 9,000 mg')
legend()
title('Cost per player at each sodium limit')
show()

saltResults.plot(x='Sodium limit (mg)', y='Shadow price of salt ($ per 100 mg)', color='darkorange')
axvline(base_limit, color='red', linewidth=3, label='base: 9,000 mg')
legend()
title('The shadow price of salt')
show()

**The red line is the base case:** the 9,000 mg limit in this assignment.

**How to read the second picture,** from right (loose limit) to left (tight limit):

- **Flat at \$0:** the sodium rule is **not binding**. The cheapest order does not use all the sodium it
  is allowed, so allowing more saves nothing.
- **Steps up:** now the rule **is binding**. Each 100 mg you take away pushes the order toward less
  salty, more expensive food.
- **The line stops:** no order is possible below that limit (infeasible).

## 5.3 · Read the shadow price at three limits

`shadow_price_at(...)` reads two rows of your table and does the subtraction for you, in words.
Run the three cells. The first two are done for you; in the third you can choose the limit.

In [ ]:
shadow_price_at(9000)      # the limit in this assignment

In [ ]:
shadow_price_at(7000)      # the limit we used in class

In [ ]:
my_limit = lowest_limit + 500      # <-- CHANGE if you like: any limit from your table (a multiple of 100)

shadow_price_at(my_limit)

### YOUR ANSWER - Step 5

**5a (8 points).** The lowest sodium limit my restaurant can meet is \_\_\_\_\_\_\_\_ mg. Below that, the table
shows \_\_\_\_\_\_, which means: (one sentence)

*your sentence*

**5b (10 points).** At the 9,000 mg limit, the shadow price of salt at my restaurant is \$ \_\_\_\_\_\_ per
player, per 100 mg. Is the sodium rule binding at 9,000 mg? \_\_\_\_\_ (yes or no). How can you tell?

*your sentence*

**5c (10 points).** Look at what `shadow_price_at(7000)` printed. In **your own words**, tell the team
doctor what it means. Use dollars and milligrams in your sentence. (If no order is possible at 7,000 mg,
tell the doctor that, and what limit is the lowest that works.)

*your sentence*

**5d (9 points).** I looked at a limit of \_\_\_\_\_\_\_\_ mg. The shadow price there is \$ \_\_\_\_\_\_ per
player, per 100 mg. Is that higher or lower than at 9,000 mg, and why does that make sense?

*your sentence*

**5e (8 points).** Use your table. Going from the 9,000 mg limit down to \_\_\_\_\_\_\_\_ mg (the limit you
looked at in 5d) changes the cost per player from \$ \_\_\_\_\_\_ to \$ \_\_\_\_\_\_. For the team of 53 that
is \$ \_\_\_\_\_\_ more per day.

# Step 6 · Your recommendation (15 points)

The athletic director has never heard of Pyomo. In **four or five sentences**, tell them:

1. what the cheapest order costs per player,
2. what sodium limit you recommend, and
3. what that limit costs, using a shadow price or a cost from Step 5.

### YOUR ANSWER - Step 6

*your four or five sentences*

# Turn it in

1. Check that Step 1 has **your name** and **your restaurant's** file name (not McDonald's).
2. Click **Runtime → Run all** and wait for it to finish. Scroll down and make sure every step printed
   something and there are no red error messages.
3. Check that every **YOUR ANSWER** cell is filled in.
4. **File → Download → Download .ipynb**, and upload that file to HuskyCT.

**Stuck?** The most common problem is a typo in a file name or an item name. Copy and paste the name
instead of typing it.